# Compare position-matrix sources: `.mmn` vs `_r_full.dat`

Compares the two position-matrix constructions at the matrix and $A(k)$ level.

- **`mmn`**: $\langle 0i|r|Rj\rangle$ rebuilt in house from the `.mmn` overlaps: every neighbour link is given the right-centre phase $\exp(i\,b\cdot\tau_j)$, the links are summed into one finite-difference derivative, and the Fourier coefficients are then mapped with $\min|R+\tau_j-\tau_i|$.
- **`rfull`**: postw90's `_r_full.dat` written with `transl_inv_full=T` and `use_ws_distance=T`: each link carries the *symmetric* phase $\exp(i\,b\cdot(\tau_i+\tau_j)/2)$ and is kept separate through the Fourier transform, with the Wigner-Seitz mapping applied afterwards.

The two orderings do not commute, so $A(R)$ is expected to differ. The question this notebook answers is *how*: a pure choice of aliasing-class representative leaves $A(k)$ untouched on the ab-initio mesh and moves it only between those points, whereas a different finite-difference discretisation changes $A(k)$ everywhere -- including at the source k-points, where both constructions see exactly the same ab-initio overlaps. That distinction is the whole argument for or against swapping the routes: only the second kind of difference can change a converged physical observable.

The original CLI script only printed its report; this notebook also saves it to `results/position_matrix_sources_mmn_vs_rfull/` so `RUN_COMPUTATION = False` has something to reload.

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
from pythtb import W90

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "validation").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from validation._paths import (
    RESULTS,
    STO_AA_CACHE,
    STO_BASE_RUN,
    STO_TI_Q_01A_RUN,
)
from validation._paths import DATA
from wannier.wannier_io import read_wannier_checkpoint
from wannier.position_matrix import (
    apply_orbital_dependent_R_mapping,
    build_position_matrix_from_mmn,
    install_postw90_position_matrix,
)

## Config

In [ ]:
RUN_COMPUTATION = False  # True: rebuild and recompute. False: load results/position_matrix_sources_mmn_vs_rfull/.
OUTPUT_DIR = RESULTS / "position_matrix_sources_mmn_vs_rfull"

PREFIX = "SrTiO3"
BASE = STO_BASE_RUN
MODE = STO_TI_Q_01A_RUN
AA_CACHE = STO_AA_CACHE
# The .mmn belongs to the *structure*, not to the projection set, so both trial
# sets of one structure share the same overlap file.
MODE_MMN = DATA / "SrTiO3/Ti_Q_01A/output/181839bc889/trial_03_Sr_sp_Ti_pd_Osp/no_displacement/185119bc889"

STRUCTURES = {
    "c_base": {
        "w90": DATA / "SrTiO3/base/soc/output/181642bc889/trial_03_Sr_sp_Ti_pd_O_sp/185116bc889",
        "mmn": BASE,
        "cache": DATA / "SrTiO3/_AA_cache/AA_trial_03_Sr_sp_Ti_pd_O_sp_base_185116bc889_wbnone_ws1em05.npz",
    },
    "c_mode": {
        "w90": MODE_MMN,
        "mmn": MODE_MMN,
        "cache": DATA / "SrTiO3/_AA_cache/AA_trial_03_Sr_sp_Ti_pd_O_sp_mode_185119bc889_wbnone_ws1em05.npz",
    },
    "b_base": {
        "w90": DATA / "SrTiO3/base/soc/output/181642bc889/trial_02_Sr_p_O_sp/183813bc889",
        "mmn": BASE,
        "cache": DATA / "SrTiO3/_AA_cache/AA_trial_02_Sr_p_O_sp_base_183813bc889_wbnone_ws1em05.npz",
    },
    "b_mode": {
        "w90": DATA / "SrTiO3/Ti_Q_01A/output/181839bc889/trial_02_Sr_p_O_sp/no_displacement/183965bc889",
        "mmn": MODE_MMN,
        "cache": DATA / "SrTiO3/_AA_cache/AA_trial_02_Sr_p_O_sp_mode_183965bc889_wbnone_ws1em05.npz",
    },
}
STRUCTURES_TO_RUN = list(STRUCTURES)  # was --structures
N_RANDOM = 64   # was --n-random: off-mesh k-points sampled for the interpolation test
SEED = 0        # was --seed

## Functions

In [ ]:
def stack(position_R: dict, R_list) -> np.ndarray:
    """``(n_R, 3, nw, nw)`` in the order of ``R_list``."""
    return np.stack([position_R[R] for R in R_list], axis=0)


def A_of_k(stacked: np.ndarray, R_list, kpoints: np.ndarray) -> np.ndarray:
    """``A(k) = sum_R exp(2 pi i k.R) A(R)`` for reduced ``k``."""
    R = np.asarray(R_list, dtype=float)
    phases = np.exp(2j * np.pi * (np.asarray(kpoints, dtype=float) @ R.T))
    return np.einsum("kr,raij->kaij", phases, stacked)


def relative(difference: np.ndarray, reference: np.ndarray) -> float:
    denominator = float(np.linalg.norm(reference))
    return float(np.linalg.norm(difference) / denominator) if denominator else np.nan


def build_model_with_source(directory: Path, mp_grid, setup: dict, source: str, prefix: str = PREFIX):
    """Returns (model, A_R) for one of the two position-matrix sources."""
    w90 = W90(str(directory), prefix)
    apply_orbital_dependent_R_mapping(w90, mp_grid, verbose=False)
    model = w90.model(zero_energy=0.0, min_hopping_norm=0.0)
    if source == "mmn":
        build_position_matrix_from_mmn(
            model, directory, prefix,
            mmn_directory=Path(setup["mmn"]),
            cache=Path(setup["cache"]),
            translational_invariant_JM=False,
            translational_invariant_MV=False,
            R_distance_tolerance=1e-5,
        )
    else:
        rfull_path = directory / f"{prefix}_r_full.dat"
        if not rfull_path.is_file():
            raise FileNotFoundError(rfull_path)
        centres = (
            np.asarray(w90.lattice.orb_vecs, dtype=float)
            @ np.asarray(w90.lat, dtype=float)
        )
        install_postw90_position_matrix(model, rfull_path, centers_cartesian=centres)
    return model, model._pos_r


def compare(structure: str, n_random: int, seed: int) -> dict:
    setup = STRUCTURES[structure]
    directory = Path(setup["w90"])
    mp_grid = tuple(
        int(v) for v in read_wannier_checkpoint(directory / f"{PREFIX}.chk")["mp_grid"]
    )

    _, A_mmn = build_model_with_source(directory, mp_grid, setup, "mmn")
    _, A_rfull = build_model_with_source(directory, mp_grid, setup, "rfull")

    shared = sorted(set(A_mmn) & set(A_rfull))
    only_mmn = sorted(set(A_mmn) - set(A_rfull))
    only_rfull = sorted(set(A_rfull) - set(A_mmn))

    stacked_mmn = stack(A_mmn, shared)
    stacked_rfull = stack(A_rfull, shared)
    difference = stacked_rfull - stacked_mmn

    num_wann = stacked_mmn.shape[-1]
    diagonal = np.arange(num_wann)
    diagonal_difference = difference[:, :, diagonal, diagonal]
    off_diagonal = difference.copy()
    off_diagonal[:, :, diagonal, diagonal] = 0.0

    # The ab-initio mesh: both constructions consume exactly the same overlaps
    # here, so any disagreement is discretisation, not representative choice.
    axes = [np.arange(n) / n for n in mp_grid]
    source_mesh = np.stack(np.meshgrid(*axes, indexing="ij"), axis=-1).reshape(-1, 3)
    generator = np.random.default_rng(seed)
    off_mesh = generator.random((n_random, 3))

    A_source_mmn = A_of_k(stacked_mmn, shared, source_mesh)
    A_source_difference = A_of_k(difference, shared, source_mesh)
    A_off_mmn = A_of_k(stacked_mmn, shared, off_mesh)
    A_off_difference = A_of_k(difference, shared, off_mesh)

    return {
        "structure": structure,
        "directory": str(directory),
        "mp_grid": mp_grid,
        "num_wann": num_wann,
        "n_R_shared": len(shared),
        "n_R_only_mmn": len(only_mmn),
        "n_R_only_rfull": len(only_rfull),
        "R_relative": relative(difference, stacked_mmn),
        "R_max_abs": float(np.abs(difference).max()),
        "R_relative_diagonal": relative(
            diagonal_difference, stacked_mmn[:, :, diagonal, diagonal]
        ),
        "R_max_abs_diagonal": float(np.abs(diagonal_difference).max()),
        "R_relative_offdiagonal": relative(off_diagonal, stacked_mmn),
        "k_source_relative": relative(A_source_difference, A_source_mmn),
        "k_source_max_abs": float(np.abs(A_source_difference).max()),
        "k_off_relative": relative(A_off_difference, A_off_mmn),
        "k_off_max_abs": float(np.abs(A_off_difference).max()),
    }


def report(result: dict) -> None:
    print(f"=== {result['structure']}: {result['directory']}")
    print(
        f"  num_wann = {result['num_wann']}, mp_grid = {result['mp_grid']}, "
        f"shared R = {result['n_R_shared']} "
        f"(mmn-only {result['n_R_only_mmn']}, rfull-only {result['n_R_only_rfull']})"
    )
    print("  A(R)  rfull - mmn")
    print(f"    all          : rel {result['R_relative']:.3e}   max {result['R_max_abs']:.3e} A")
    print(f"    diagonal     : rel {result['R_relative_diagonal']:.3e}   max {result['R_max_abs_diagonal']:.3e} A")
    print(f"    off-diagonal : rel {result['R_relative_offdiagonal']:.3e}")
    print("  A(k)  rfull - mmn")
    print(f"    ab-initio mesh : rel {result['k_source_relative']:.3e}   max {result['k_source_max_abs']:.3e} A")
    print(f"    off-mesh       : rel {result['k_off_relative']:.3e}   max {result['k_off_max_abs']:.3e} A")
    if result["k_source_relative"] < 1e-10:
        verdict = (
            "A(k) agrees on the ab-initio mesh: the two A(R) are different "
            "representatives of ONE interpolant."
        )
    else:
        verdict = (
            "A(k) already differs on the ab-initio mesh: the two are different "
            "finite-difference discretisations, not different representatives."
        )
    print(f"  -> {verdict}\n")

## Run + save

`results[structure]` holds the full comparison dict as a live object -- e.g. `results["c_base"]["R_relative"]`.

In [ ]:
if RUN_COMPUTATION:
    results = {}
    for structure in STRUCTURES_TO_RUN:
        results[structure] = compare(structure, N_RANDOM, SEED)
        report(results[structure])
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / "results.json").write_text(json.dumps(results, indent=2, sort_keys=True) + "\n")
    print(f"saved {OUTPUT_DIR / 'results.json'}")

## Load a previous run (skip if you just computed above)

In [ ]:
if not RUN_COMPUTATION:
    results_path = OUTPUT_DIR / "results.json"
    if not results_path.exists():
        raise FileNotFoundError(
            f"{results_path} does not exist yet -- this check has never been run and "
            "saved. Set RUN_COMPUTATION = True above and rerun from the config cell down "
            "to populate it once."
        )
    results = json.loads(results_path.read_text())
    for structure, result in results.items():
        report(result)